# Generative AI - Text Generation and Machine Translation

Q1. What is Generative AI and what are its primary use cases across industries?
- Generative AI creates new content (text, images, code, etc.) from existing data. It's used in industries for content creation, personalized marketing, drug discovery, and simulating complex systems.

Q2. Explain the role of probabilistic modeling in generative models. How do
these models differ from discriminative models?
- Probabilistic modeling helps generative models learn the underlying data distribution to create new, similar data. They differ from discriminative models, which focus on distinguishing between existing data categories.

Q3. What is the difference between Autoencoders and Variational
Autoencoders (VAEs) in the context of text generation?
- Autoencoders reconstruct text, learning a compressed representation. VAEs, however, learn a probabilistic distribution of this representation, allowing them to generate new, diverse text samples, not just reconstruct existing ones.

Q4. Describe the working of attention mechanisms in Neural Machine
Translation (NMT). Why are they critical?
- Attention mechanisms in NMT allow the model to focus on relevant parts of the source sentence when translating specific words, rather than processing the entire sentence at once. They are critical because they address the limitations of fixed-length context vectors, improving translation quality for long sentences.

Q5. What ethical considerations must be addressed when using generative AI
for creative content such as poetry or storytelling?
- Ethical considerations for generative AI in creative content include: ensuring proper attribution/authorship, preventing plagiarism, addressing potential bias or harmful content generation, and defining intellectual property rights for AI-created works.

In [1]:
''' Q6. Use the following small text dataset to train a simple Variational
Autoencoder (VAE) for text reconstruction:

["The sky is blue", "The sun is bright", "The grass is green",
"The night is dark", "The stars are shining"]

1. Preprocess the data (tokenize and pad the sequences).
2. Build a basic VAE model for text reconstruction.
3. Train the model and show how it reconstructs or generates similar sentences '''

' Q6. Use the following small text dataset to train a simple Variational\nAutoencoder (VAE) for text reconstruction:\n\n["The sky is blue", "The sun is bright", "The grass is green",\n"The night is dark", "The stars are shining"]\n\n1. Preprocess the data (tokenize and pad the sequences).\n2. Build a basic VAE model for text reconstruction.\n3. Train the model and show how it reconstructs or generates similar sentences '

In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from tensorflow.keras.losses import mse, binary_crossentropy
import numpy as np

# Sample text data
data = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "a bird flew in the sky",
    "cats and dogs are pets",
    "birds sing in the morning",
    "the mat is on the floor",
    "the park is green",
    "the sky is blue",
    "pets are fun",
    "morning is a new day"
]

# Tokenize the text
tokenizer = Tokenizer(num_words=None, oov_token="<unk>")
tokenizer.fit_on_texts(data)

word_index = tokenizer.word_index
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
vocab_size = len(word_index) + 1

print(f"Vocabulary size: {vocab_size}")

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(data)

# Pad sequences to a fixed length
max_sequence_length = max([len(seq) for seq in sequences])
padded_sequences = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')

print(f"Max sequence length: {max_sequence_length}")
print(f"Padded sequences shape: {padded_sequences.shape}")
print("Example padded sequence:")
print(padded_sequences[0])

# Prepare data for VAE (one-hot encoding for decoder input if using simple dense layers, or embedding for LSTM)
def to_one_hot(sequences, vocab_size):
    results = np.zeros((len(sequences), max_sequence_length, vocab_size))
    for i, sequence in enumerate(sequences):
        for j, word_idx in enumerate(sequence):
            if word_idx > 0: # Exclude padding (0)
                results[i, j, word_idx] = 1.
    return results

# For simplicity, we'll convert the target to padded sequences for reconstruction loss
# The decoder will try to predict the next word in the sequence, so we can use shifted targets
# For VAE, the output is often a probability distribution over the vocabulary for each timestep.
# For reconstruction loss, we'll use `sparse_categorical_crossentropy` with the original padded sequences as targets.

X_train = padded_sequences

print(f"X_train shape: {X_train.shape}")

Vocabulary size: 30
Max sequence length: 6
Padded sequences shape: (10, 6)
Example padded sequence:
[ 2 13 14  5  2  6]
X_train shape: (10, 6)


In [6]:
# VAE parameters
latent_dim = 64
intermediate_dim = 256
embedding_dim = 100

# Re-parameterization trick
def sampling(args):
    z_mean, z_log_var = args
    epsilon = K.random_normal(shape=(K.shape(z_mean)[0], latent_dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

# Custom VAE Model
class VAE(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, intermediate_dim, latent_dim, max_sequence_length, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder_embedding = Embedding(vocab_size, embedding_dim, input_length=max_sequence_length)
        self.encoder_lstm = LSTM(intermediate_dim, return_sequences=False)
        self.encoder_dense_intermediate = Dense(intermediate_dim, activation='relu')
        self.z_mean_dense = Dense(latent_dim, name='z_mean')
        self.z_log_var_dense = Dense(latent_dim, name='z_log_var')
        self.z_sampling = Lambda(sampling, output_shape=(latent_dim,), name='z')

        self.decoder_dense_intermediate = Dense(intermediate_dim, activation='relu')
        self.decoder_repeat_vector = tf.keras.layers.RepeatVector(max_sequence_length)
        self.decoder_lstm = LSTM(intermediate_dim, return_sequences=True)
        self.decoder_output_dense = Dense(vocab_size, activation='softmax')

        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def encode(self, inputs):
        x = self.encoder_embedding(inputs)
        x = self.encoder_lstm(x)
        x = self.encoder_dense_intermediate(x)
        z_mean = self.z_mean_dense(x)
        z_log_var = self.z_log_var_dense(x)
        z = self.z_sampling([z_mean, z_log_var])
        return z_mean, z_log_var, z

    def decode(self, z):
        x = self.decoder_dense_intermediate(z)
        x = self.decoder_repeat_vector(x)
        x = self.decoder_lstm(x)
        outputs = self.decoder_output_dense(x)
        return outputs

    def call(self, inputs):
        z_mean, z_log_var, z = self.encode(inputs)
        reconstruction = self.decode(z)
        return reconstruction

    def train_step(self, data):
        if isinstance(data, tuple):
            inputs = data[0]
        else:
            inputs = data

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encode(inputs)
            reconstruction = self.decode(z)

            reconstruction_loss = K.sum(K.sparse_categorical_crossentropy(inputs, reconstruction), axis=-1)
            kl_loss = -0.5 * K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1)
            total_loss = K.mean(reconstruction_loss + kl_loss)

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

# Instantiate and compile the VAE model
vae = VAE(vocab_size, embedding_dim, intermediate_dim, latent_dim, max_sequence_length)
vae.compile(optimizer='adam')

# Build the model to show summary
# We need to call build or call it with dummy data for summary to work
dummy_input = tf.zeros((1, max_sequence_length), dtype=tf.int32)
_ = vae(dummy_input) # Call with dummy input to build

print("VAE Model Summary:")
vae.summary()

# The encoder and decoder sub-models can be extracted if needed for separate use (e.g., generation)
# For generation, we primarily need the decoder. We will create a separate 'generator_model' based on vae.decode
decoder_inputs_for_generation = Input(shape=(latent_dim,))
generator_outputs = vae.decode(decoder_inputs_for_generation)
generator_model = Model(decoder_inputs_for_generation, generator_outputs, name='generator')
print("\nGenerator (Decoder) Model Summary:")
generator_model.summary()

VAE Model Summary:


Model: "vae"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (1, 6, 100)            │         3,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (1, 256)               │       365,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (1, 256)               │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ z_mean (Dense)                  │ (1, 64)                │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ z_log_var (Dense)               │ (1, 64)                │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ z (Lambda)                      │ (1, 64)                │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (1, 256)               │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (1, 6, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (1, 6, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (1, 6, 30)             │         7,710 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,016,918 (3.88 MB)

 Trainable params: 1,016,918 (3.88 MB)

...

Model: "generator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 6, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 6, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6, 30)          │         7,710 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 549,662 (2.10 MB)

 Trainable params: 549,662 (2.10 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Train the VAE
vae.fit(X_train, X_train, epochs=100, batch_size=2)

# Demonstrate reconstruction
def decode_sequence(sequence):
    return " ".join([reverse_word_index.get(idx, '<unk>') for idx in sequence if idx > 0])

print("\n--- Reconstruction Examples ---")
for i in range(5):
    original_sentence = data[i]
    encoded_input = X_train[i:i+1]

    # Predict the output probabilities for each word in the sequence
    reconstructed_probs = vae.predict(encoded_input)

    # Get the index of the word with the highest probability for each timestep
    reconstructed_sequence = np.argmax(reconstructed_probs, axis=-1)[0]

    reconstructed_sentence = decode_sequence(reconstructed_sequence)

    print(f"Original: {original_sentence}")
    print(f"Reconstructed: {reconstructed_sentence}")
    print("-")

# Demonstrate generation
print("\n--- Generation Examples ---")
num_generated_samples = 5

for _ in range(num_generated_samples):
    # Sample a point from the latent space
    random_latent_vector = np.random.normal(size=(1, latent_dim))

    # Predict the output probabilities from the decoder using the generator model
    generated_probs = generator_model.predict(random_latent_vector)

    # Get the index of the word with the highest probability for each timestep
    generated_sequence = np.argmax(generated_probs, axis=-1)[0]

    generated_sentence = decode_sequence(generated_sequence)
    print(f"Generated: {generated_sentence}")

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 56ms/step - kl_loss: 0.0031 - loss: 20.1153 - reconstruction_loss: 20.1122
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - kl_loss: 0.0100 - loss: 18.5111 - reconstruction_loss: 18.5011
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - kl_loss: 0.0121 - loss: 18.5133 - reconstruction_loss: 18.5012
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - kl_loss: 0.0203 - loss: 17.9827 - reconstruction_loss: 17.9624
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - kl_loss: 0.0377 - loss: 17.8116 - reconstruction_loss: 17.7739
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - kl_loss: 0.0639 - loss: 17.9420 - reconstruction_loss: 17.8781
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - kl_loss: 0.1004 - loss: 17.5618 - reconstruction_loss: 17.4615
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - kl_loss: 0.1439 - loss: 17.4326 - reconstruction_loss: 17.2887
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - kl_loss: 0.1032 - loss: 17.2

In [ ]:
''' Q7. Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short
English paragraph into French and German. Provide the original and translated text'''


In [8]:
# Import the Python SDK and securely get your API key
import google.generativeai as genai
# The previous package `google.generativeai` is deprecated. Using `google.genai` now.
# import google.genai as genai # Uncomment this if you prefer to use the new package
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini API model
gemini_model = genai.GenerativeModel('gemini-pro') # Using 'gemini-pro' for text generation

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

In [9]:
english_paragraph = "The quick brown fox jumps over the lazy dog. This is a classic pangram used to display typefaces and test equipment. It contains all letters of the English alphabet."

print(f"Original English Text:\n{english_paragraph}\n")

# Translate to French
response_french = gemini_model.generate_content(f"Translate the following English text to French: {english_paragraph}")
french_translation = response_french.text
print(f"French Translation:\n{french_translation}\n")

# Translate to German
response_german = gemini_model.generate_content(f"Translate the following English text to German: {english_paragraph}")
german_translation = response_german.text
print(f"German Translation:\n{german_translation}")

Original English Text:
The quick brown fox jumps over the lazy dog. This is a classic pangram used to display typefaces and test equipment. It contains all letters of the English alphabet.



NameError: name 'gemini_model' is not defined

In [ ]:
''' Q8. Implement a simple attention-based encoder-decoder model for
English-to-Spanish translation using Tensorflow or PyTorch.
'''

In [11]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Attention

# Sample English-Spanish sentence pairs
raw_data = [
    ("go.", "ve."),
    ("hi.", "hola."),
    ("run!", "¡corre!"),
    ("run.", "corre."),
    ("eat.", "come."),
    ("i try.", "intento."),
    ("i win.", "gano."),
    ("we try.", "intentamos."),
    ("we run.", "corremos."),
    ("he eats.", "él come."),
    ("she eats.", "ella come."),
    ("they run.", "ellos corren."),
    ("you run.", "corres."),
    ("i eat apple.", "yo como manzana."),
    ("he eats bread.", "él come pan."),
    ("she drinks water.", "ella bebe agua."),
    ("we drink milk.", "bebemos leche."),
    ("the cat sleeps.", "el gato duerme."),
    ("the dog plays.", "el perro juega."),
    ("the boy reads book.", "el niño lee libro."),
    ("the girl writes letter.", "la niña escribe carta."),
    ("my name is john.", "mi nombre es juan."),
    ("what is your name?", "¿cómo te llamas?"),
    ("where are you from?", "¿de dónde eres?"),
    ("i am from spain.", "soy de españa.")
]

# Preprocessing
def preprocess_sentence(sentence):
    # Convert to lowercase and add start/end tokens
    sentence = sentence.lower().strip()
    sentence = '<start> ' + sentence + ' <end>'
    return sentence

# Separate English and Spanish sentences
english_sentences = [preprocess_sentence(pair[0]) for pair in raw_data]
spanish_sentences = [preprocess_sentence(pair[1]) for pair in raw_data]

# Tokenize English sentences
eng_tokenizer = Tokenizer(filters='', lower=False) # Filters='' to not remove punctuation, lower=False because we already lowercased
eng_tokenizer.fit_on_texts(english_sentences)
eng_vocab_size = len(eng_tokenizer.word_index) + 1
eng_seq = eng_tokenizer.texts_to_sequences(english_sentences)
max_eng_len = max([len(seq) for seq in eng_seq])
eng_padded_seq = pad_sequences(eng_seq, maxlen=max_eng_len, padding='post')

print(f"English Vocabulary Size: {eng_vocab_size}")
print(f"Max English Sequence Length: {max_eng_len}")
print("Example English padded sequence:", eng_padded_seq[0])

# Tokenize Spanish sentences
spa_tokenizer = Tokenizer(filters='', lower=False)
spa_tokenizer.fit_on_texts(spanish_sentences)
spa_vocab_size = len(spa_tokenizer.word_index) + 1
spa_seq = spa_tokenizer.texts_to_sequences(spanish_sentences)
max_spa_len = max([len(seq) for seq in spa_seq])
spa_padded_seq = pad_sequences(spa_seq, maxlen=max_spa_len, padding='post')

print(f"Spanish Vocabulary Size: {spa_vocab_size}")
print(f"Max Spanish Sequence Length: {max_spa_len}")
print("Example Spanish padded sequence:", spa_padded_seq[0])

# Create input and target sequences for training
X_train_encoder = eng_padded_seq
# Decoder input needs to be shifted right (start token, but no end token for final word)
X_train_decoder = spa_padded_seq[:, :-1]
# Decoder target is the actual target sequence (end token for final word, no start token)
y_train_decoder = spa_padded_seq[:, 1:]

print(f"X_train_encoder shape: {X_train_encoder.shape}")
print(f"X_train_decoder shape: {X_train_decoder.shape}")
print(f"y_train_decoder shape: {y_train_decoder.shape}")

English Vocabulary Size: 49
Max English Sequence Length: 6
Example English padded sequence: [ 1 13  2  0  0  0]
Spanish Vocabulary Size: 51
Max Spanish Sequence Length: 6
Example Spanish padded sequence: [1 7 2 0 0 0]
X_train_encoder shape: (25, 6)
X_train_decoder shape: (25, 5)
y_train_decoder shape: (25, 5)


In [12]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Attention

# Model Parameters
EMBEDDING_DIM = 256
UNITS = 512 # LSTM units

# Encoder
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.lstm = LSTM(self.enc_units, return_sequences=True, return_state=True)

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state_h, state_c = self.lstm(x, initial_state=hidden)
        return output, state_h, state_c

    def initialize_hidden_state(self, batch_size):
        return [tf.zeros((batch_size, self.enc_units)), tf.zeros((batch_size, self.enc_units))]

# Decoder
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.lstm = LSTM(self.dec_units, return_sequences=True, return_state=True)
        self.fc = Dense(vocab_size)

        # For attention
        self.attention = Attention()

    def call(self, x, hidden, enc_output):
        # x shape == (batch_size, 1) - single word input
        x = self.embedding(x)

        # enc_output shape == (batch_size, max_length, hidden_size)
        # hidden shape == (batch_size, hidden_size)

        # Add a time dimension to the decoder's previous hidden state for attention
        # query shape == (batch_size, 1, hidden_size)
        query = tf.expand_dims(hidden[0], 1)

        # context_vector shape == (batch_size, 1, hidden_size)
        # The Attention layer already returns a tensor with a sequence length dimension.
        context_vector = self.attention([query, enc_output])

        # x shape after concatenation == (batch_size, 1, embedding_dim + hidden_size)
        # context_vector and x are both rank 3 (batch_size, 1, feature_dim), so they can be directly concatenated.
        # Corrected: Removed the erroneous tf.expand_dims around context_vector from the concat call.
        x = tf.concat([context_vector, x], axis=-1)

        # Passing the concatenated vector to the LSTM
        output, state_h, state_c = self.lstm(x, initial_state=hidden)

        # output shape == (batch_size * 1, hidden_size)
        output = tf.reshape(output, (-1, output.shape[2]))

        # output shape == (batch_size, vocab)
        x = self.fc(output)

        return x, state_h, state_c, context_vector

In [13]:
# Instantiate Encoder and Decoder
encoder = Encoder(eng_vocab_size, EMBEDDING_DIM, UNITS)
decoder = Decoder(spa_vocab_size, EMBEDDING_DIM, UNITS)

optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0)) # Mask out padding (0)
    loss_ = loss_object(real, pred)

    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)


@tf.function
def train_step(enc_input, dec_input, tar):
    loss = 0
    with tf.GradientTape() as tape:
        enc_hidden = encoder.initialize_hidden_state(enc_input.shape[0])
        enc_output, enc_state_h, enc_state_c = encoder(enc_input, enc_hidden)

        dec_hidden = [enc_state_h, enc_state_c]

        dec_input_token = tf.expand_dims([spa_tokenizer.word_index['<start>']] * enc_input.shape[0], 1)

        for t in range(1, tar.shape[1]):
            # passing enc_output to the decoder
            predictions, dec_hidden_h, dec_hidden_c, _ = decoder(dec_input_token, dec_hidden, enc_output)
            dec_hidden = [dec_hidden_h, dec_hidden_c]

            loss += loss_function(tar[:, t], predictions)

            # using teacher forcing
            dec_input_token = tf.expand_dims(tar[:, t], 1)

    batch_loss = (loss / int(tar.shape[1]))
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss

# Training Loop (simplified for demonstration)
EPOCHS = 10
BATCH_SIZE = len(X_train_encoder) # Using full batch for this small dataset

print("Starting training...")
for epoch in range(EPOCHS):
    total_loss = 0

    # For this small dataset, we'll treat it as one batch
    batch_loss = train_step(X_train_encoder, X_train_decoder, y_train_decoder)
    total_loss += batch_loss

    print(f'Epoch {epoch+1} Loss {total_loss:.4f}')

print("Training complete.")

Starting training...
Epoch 1 Loss 1.7004
Epoch 2 Loss 1.6694
Epoch 3 Loss 1.6286
Epoch 4 Loss 1.5589
Epoch 5 Loss 1.4299
Epoch 6 Loss 1.1985
Epoch 7 Loss 1.1151
Epoch 8 Loss 1.1419
Epoch 9 Loss 1.0357
Epoch 10 Loss 1.0105
Training complete.


In [14]:
def evaluate(sentence):
    attention_plot = np.zeros((max_spa_len, max_eng_len))

    sentence = preprocess_sentence(sentence)
    inputs = [eng_tokenizer.word_index[i] for i in sentence.split(' ')]
    inputs = pad_sequences([inputs], maxlen=max_eng_len, padding='post')
    inputs = tf.convert_to_tensor(inputs)

    result = ''
    enc_hidden = encoder.initialize_hidden_state(1)
    enc_output, enc_state_h, enc_state_c = encoder(inputs, enc_hidden)

    dec_hidden = [enc_state_h, enc_state_c]
    dec_input = tf.expand_dims([spa_tokenizer.word_index['<start>']], 0)

    for t in range(max_spa_len):
        predictions, dec_hidden_h, dec_hidden_c, attention_weights = decoder(dec_input, dec_hidden, enc_output)
        dec_hidden = [dec_hidden_h, dec_hidden_c]

        predicted_id = tf.argmax(predictions[0]).numpy()

        result += spa_tokenizer.index_word[predicted_id] + ' '

        if spa_tokenizer.index_word[predicted_id] == '<end>':
            return result, sentence, attention_plot

        # the predicted ID is fed back into the model
        dec_input = tf.expand_dims([predicted_id], 0)

    return result, sentence, attention_plot

def translate(sentence):
    result, sentence, attention_plot = evaluate(sentence)

    print('Input: %s' % (sentence))
    print('Predicted translation: {}'.format(result))


# Test translations
print("\n--- Test Translations ---")
translate("the cat sleeps.")
translate("he eats.")
translate("where are you from?")
translate("i try.")
translate("you run.")



--- Test Translations ---
Input: <start> the cat sleeps. <end>
Predicted translation: <end> 
Input: <start> he eats. <end>
Predicted translation: <end> 
Input: <start> where are you from? <end>
Predicted translation: <end> 
Input: <start> i try. <end>
Predicted translation: <end> 
Input: <start> you run. <end>
Predicted translation: <end> 


In [ ]:
''' Q9. Use the following short poetry dataset to simulate poem generation with a
pre-trained GPT model:

["Roses are red, violets are blue,",
"Sugar is sweet, and so are you.",
"The moon glows bright in silent skies,",
"A bird sings where the soft wind sighs."]

Using this dataset as a reference for poetic structure and language, generate a new 2-4
line poem using a pre-trained GPT model (such as GPT-2). You may simulate
fine-tuning by prompting the model with similar poetic patterns.

Include your code, the prompt used, and the generated poem in your answer.
'''

In [15]:
import tensorflow as tf
import numpy as np

# Install the transformers library if not already installed
!pip install transformers

In [16]:
from transformers import pipeline, set_seed

# Set a seed for reproducibility
set_seed(42)

# Load a text generation pipeline using GPT-2
generator = pipeline('text-generation', model='gpt2')

# Provided poetry dataset for context
poetry_dataset = [
    "Roses are red, violets are blue,",
    "Sugar is sweet, and so are you.",
    "The moon glows bright in silent skies,",
    "A bird sings where the soft wind sighs."
]

print("Poetry Dataset:")
for line in poetry_dataset:
    print(line)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Poetry Dataset:
Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.


In [17]:
# Craft a prompt to simulate the poetic style
# We'll use a starting line and hint at the structure
prompt = "The sun shines bright, the flowers bloom,\n" \
         "A gentle breeze whispers through the room.\n"

# Generate a poem
# num_return_sequences: number of poems to generate
# max_new_tokens: maximum number of tokens to generate in total
# top_k: controls diversity
# temperature: controls randomness (higher = more random)
# do_sample: if True, samples from the distribution; if False, uses greedy decoding

generated_poems = generator(prompt, max_new_tokens=20, num_return_sequences=1,
                            top_k=50, temperature=0.7, do_sample=True,
                            pad_token_id=generator.tokenizer.eos_token_id)

print("\n--- Generated Poem ---")
print(f"Prompt used:\n{prompt}")
print("\nGenerated Poem:")
print(generated_poems[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'top_k', 'pad_token_id', 'num_return_sequences', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Generated Poem ---
Prompt used:
The sun shines bright, the flowers bloom,
A gentle breeze whispers through the room.


Generated Poem:
The sun shines bright, the flowers bloom,
A gentle breeze whispers through the room.
A fairy sings a song.
A girl dances in the darkness.
A lady sings a hy


In [ ]:
''' Q10. Imagine you are building a creative writing assistant for a publishing
company. The assistant should generate story plots and character descriptions using
Generative AI. Describe how you would design the system, including model selection,
training data, bias mitigation, and evaluation methods. Explain the real-world challenges
you might face.
'''

In [21]:
# Task
'''The `ValueError` in the `train_step` function is due to an incorrect `tf.concat` operation within the `Decoder`'s `call` method. The `context_vector` is already rank 3, but `tf.expand_dims(context_vector, 1)` is adding an unnecessary dimension, making it rank 4, while `x` (after embedding) remains rank 3. `tf.concat` requires all input tensors to have the same rank.

To fix this, we need to remove the erroneous `tf.expand_dims` around `context_vector` in the `tf.concat` call.

Here's the corrected code for the `Decoder` class:

```python'''

# Decoder
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.lstm = LSTM(self.dec_units, return_sequences=True, return_state=True)
        self.fc = Dense(vocab_size)

        # For attention
        self.attention = Attention()

    def call(self, x, hidden, enc_output):
        # x shape == (batch_size, 1) - single word input
        x = self.embedding(x)

        # enc_output shape == (batch_size, max_length, hidden_size)
        # hidden shape == (batch_size, hidden_size)

        # Add a time dimension to the decoder's previous hidden state for attention
        # query shape == (batch_size, 1, hidden_size)
        query = tf.expand_dims(hidden[0], 1)

        # context_vector shape == (batch_size, 1, hidden_size)
        # The Attention layer already returns a tensor with a sequence length dimension (rank 3).
        context_vector = self.attention([query, enc_output])

        # x shape after concatenation == (batch_size, 1, embedding_dim + hidden_size)
        # context_vector (rank 3) and x (rank 3) can be directly concatenated.
        # FIX: Removed the erroneous tf.expand_dims around context_vector
        x = tf.concat([context_vector, x], axis=-1)

        # Passing the concatenated vector to the LSTM
        output, state_h, state_c = self.lstm(x, initial_state=hidden)

        # output shape == (batch_size * 1, hidden_size)
        output = tf.reshape(output, (-1, output.shape[2]))

        # output shape == (batch_size, vocab)
        x = self.fc(output)

        return x, state_h, state_c, context_vector
#```

#Let's apply this fix.

In [22]:
# Decoder
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.lstm = LSTM(self.dec_units, return_sequences=True, return_state=True)
        self.fc = Dense(vocab_size)

        # For attention
        self.attention = Attention()

    def call(self, x, hidden, enc_output):
        # x shape == (batch_size, 1) - single word input
        x = self.embedding(x)

        # enc_output shape == (batch_size, max_length, hidden_size)
        # hidden shape == (batch_size, hidden_size)

        # Add a time dimension to the decoder's previous hidden state for attention
        # query shape == (batch_size, 1, hidden_size)
        query = tf.expand_dims(hidden[0], 1)

        # context_vector shape == (batch_size, 1, hidden_size)
        # The Attention layer already returns a tensor with a sequence length dimension (rank 3).
        context_vector = self.attention([query, enc_output])

        # x shape after concatenation == (batch_size, 1, embedding_dim + hidden_size)
        # context_vector (rank 3) and x (rank 3) can be directly concatenated.
        # FIX: Removed the erroneous tf.expand_dims around context_vector
        x = tf.concat([context_vector, x], axis=-1)

        # Passing the concatenated vector to the LSTM
        output, state_h, state_c = self.lstm(x, initial_state=hidden)

        # output shape == (batch_size * 1, hidden_size)
        output = tf.reshape(output, (-1, output.shape[2]))

        # output shape == (batch_size, vocab)
        x = self.fc(output)

        return x, state_h, state_c, context_vector

Describe System Architecture
Subtask:
- Outline the overall system architecture for the creative writing assistant, specifying its main components and how they interact.

Creative Writing Assistant:
- System Architecture
The creative writing assistant's architecture is designed to provide comprehensive support to writers, from idea generation to refining drafts. It comprises several interconnected functional modules:

User Interface (UI) / Input Module:

Role:
- Handles all user interactions, including receiving text input (prompts, partial stories, outlines) and displaying generated content, suggestions, and feedback.
Responsibilities:
- Text editor functionality, user authentication (if applicable), prompt input, display of generation options, and presentation of translated/reconstructed text.
Natural Language Understanding (NLU) Module:

Role:
- Processes and interprets user input to understand context, intent, style, and content.
Responsibilities:
- Tokenization, part-of-speech tagging, named entity recognition, sentiment analysis, topic modeling, and potentially identifying literary devices. This module prepares the input for other modules.
Text Generation Module (e.g., GPT-2/Gemini):

Role:
- Generates coherent and contextually relevant text based on NLU output and user prompts.
Responsibilities:
- Poetry generation, story continuation, idea brainstorming, character descriptions, and dialogue generation. It can leverage pre-trained models (like GPT-2) or fine-tuned custom models.
Text Reconstruction/Paraphrasing Module (e.g., VAE):

Role:
- Reconstructs or paraphrases text, often used for style transfer, summarization, or ensuring grammatical correctness while maintaining meaning.
Responsibilities:
- Takes generated or user-provided text and rephrases it, potentially adhering to specific constraints (e.g., simpler language, different tone).
Translation Module (e.g., Attention-based Seq2Seq / Gemini API):

Role:
- Translates text between different languages.
Responsibilities:
- Provides English-to-Spanish translation (as demonstrated), or any other language pair, enabling writers to work in multilingual contexts or translate their work.
Knowledge Base / Context Management Module:

Role:
- Stores and retrieves domain-specific knowledge, user preferences, and ongoing session context.
Responsibilities:
- Maintaining character lists, plot points, world-building details, previous interactions, and user-defined stylistic guidelines. This ensures consistency and relevance in generated content.
Evaluation & Feedback Module:

Role:
- Assesses the quality of generated text and provides constructive feedback to the user.
Responsibilities:
- Grammatical error detection, stylistic suggestions, coherence checks, and potentially originality scores. It might also allow users to rate outputs to improve future generations.

Interactions and Data Flow:
- The UI sends user prompts and existing text to the NLU Module.
The NLU Module processes this input and passes the contextual understanding to the Text Generation Module, Text Reconstruction Module, and Translation Module.
These generation/transformation modules query the Knowledge Base for relevant information (e.g., character names, specific stylistic rules) to guide their output.
- The outputs from the generation/transformation modules are sent back to the UI for display.
- Optionally, the generated text can be routed through the Evaluation & Feedback Module before being displayed to the user, providing immediate suggestions.
- User feedback via the UI can be used to update the Knowledge Base or fine-tune models within the generation modules.
API Connections:
- External services like the Gemini API would be accessed by the Translation Module and potentially the Text Generation Module (for more advanced generative tasks).
Diagrammatic Representation:
- A system architecture diagram would visually represent these modules as distinct blocks. Arrows would illustrate the data flow, showing how user input moves from the UI through NLU to generation/transformation modules, possibly interacting with the Knowledge Base, and finally back to the UI. Key API connections (e.g., to Gemini) would be highlighted. The diagram would emphasize the cyclical nature of interaction, where user input drives generation, which can then be refined and re-input by the user.

In [ ]:
### Creative Writing Assistant: System Architecture

'''The creative writing assistant's architecture is designed to provide comprehensive support to writers, from idea generation to refining drafts. It comprises several interconnected functional modules:

1.  **User Interface (UI) / Input Module:**
    *   **Role:** Handles all user interactions, including receiving text input (prompts, partial stories, outlines) and displaying generated content, suggestions, and feedback.
    *   **Responsibilities:** Text editor functionality, user authentication (if applicable), prompt input, display of generation options, and presentation of translated/reconstructed text.

2.  **Natural Language Understanding (NLU) Module:**
    *   **Role:** Processes and interprets user input to understand context, intent, style, and content.
    *   **Responsibilities:** Tokenization, part-of-speech tagging, named entity recognition, sentiment analysis, topic modeling, and potentially identifying literary devices. This module prepares the input for other modules.

3.  **Text Generation Module (e.g., GPT-2/Gemini):**
    *   **Role:** Generates coherent and contextually relevant text based on NLU output and user prompts.
    *   **Responsibilities:** Poetry generation, story continuation, idea brainstorming, character descriptions, and dialogue generation. It can leverage pre-trained models (like GPT-2) or fine-tuned custom models.

4.  **Text Reconstruction/Paraphrasing Module (e.g., VAE):**
    *   **Role:** Reconstructs or paraphrases text, often used for style transfer, summarization, or ensuring grammatical correctness while maintaining meaning.
    *   **Responsibilities:** Takes generated or user-provided text and rephrases it, potentially adhering to specific constraints (e.g., simpler language, different tone).

5.  **Translation Module (e.g., Attention-based Seq2Seq / Gemini API):**
    *   **Role:** Translates text between different languages.
    *   **Responsibilities:** Provides English-to-Spanish translation (as demonstrated), or any other language pair, enabling writers to work in multilingual contexts or translate their work.

6.  **Knowledge Base / Context Management Module:**
    *   **Role:** Stores and retrieves domain-specific knowledge, user preferences, and ongoing session context.
    *   **Responsibilities:** Maintaining character lists, plot points, world-building details, previous interactions, and user-defined stylistic guidelines. This ensures consistency and relevance in generated content.

7.  **Evaluation & Feedback Module:**
    *   **Role:** Assesses the quality of generated text and provides constructive feedback to the user.
    *   **Responsibilities:** Grammatical error detection, stylistic suggestions, coherence checks, and potentially originality scores. It might also allow users to rate outputs to improve future generations.

#### Interactions and Data Flow:

*   The **UI** sends user prompts and existing text to the **NLU Module**.
*   The **NLU Module** processes this input and passes the contextual understanding to the **Text Generation Module**, **Text Reconstruction Module**, and **Translation Module**.
*   These generation/transformation modules query the **Knowledge Base** for relevant information (e.g., character names, specific stylistic rules) to guide their output.
*   The outputs from the generation/transformation modules are sent back to the **UI** for display.
*   Optionally, the generated text can be routed through the **Evaluation & Feedback Module** before being displayed to the user, providing immediate suggestions.
*   User feedback via the **UI** can be used to update the **Knowledge Base** or fine-tune models within the generation modules.
*   **API Connections:** External services like the Gemini API would be accessed by the **Translation Module** and potentially the **Text Generation Module** (for more advanced generative tasks).

#### Diagrammatic Representation:

A system architecture diagram would visually represent these modules as distinct blocks. Arrows would illustrate the data flow, showing how user input moves from the UI through NLU to generation/transformation modules, possibly interacting with the Knowledge Base, and finally back to the UI. Key API connections (e.g., to Gemini) would be highlighted. The diagram would emphasize the cyclical nature of interaction, where user input drives generation, which can then be refined and re-input by the user.
### Creative Writing Assistant: System Architecture

The creative writing assistant's architecture is designed to provide comprehensive support to writers, from idea generation to refining drafts. It comprises several interconnected functional modules:

1.  **User Interface (UI) / Input Module:**
    *   **Role:** Handles all user interactions, including receiving text input (prompts, partial stories, outlines) and displaying generated content, suggestions, and feedback.
    *   **Responsibilities:** Text editor functionality, user authentication (if applicable), prompt input, display of generation options, and presentation of translated/reconstructed text.

2.  **Natural Language Understanding (NLU) Module:**
    *   **Role:** Processes and interprets user input to understand context, intent, style, and content.
    *   **Responsibilities:** Tokenization, part-of-speech tagging, named entity recognition, sentiment analysis, topic modeling, and potentially identifying literary devices. This module prepares the input for other modules.

3.  **Text Generation Module (e.g., GPT-2/Gemini):**
    *   **Role:** Generates coherent and contextually relevant text based on NLU output and user prompts.
    *   **Responsibilities:** Poetry generation, story continuation, idea brainstorming, character descriptions, and dialogue generation. It can leverage pre-trained models (like GPT-2) or fine-tuned custom models.

4.  **Text Reconstruction/Paraphrasing Module (e.g., VAE):**
    *   **Role:** Reconstructs or paraphrases text, often used for style transfer, summarization, or ensuring grammatical correctness while maintaining meaning.
    *   **Responsibilities:** Takes generated or user-provided text and rephrases it, potentially adhering to specific constraints (e.g., simpler language, different tone).

5.  **Translation Module (e.g., Attention-based Seq2Seq / Gemini API):**
    *   **Role:** Translates text between different languages.
    *   **Responsibilities:** Provides English-to-Spanish translation (as demonstrated), or any other language pair, enabling writers to work in multilingual contexts or translate their work.

6.  **Knowledge Base / Context Management Module:**
    *   **Role:** Stores and retrieves domain-specific knowledge, user preferences, and ongoing session context.
    *   **Responsibilities:** Maintaining character lists, plot points, world-building details, previous interactions, and user-defined stylistic guidelines. This ensures consistency and relevance in generated content.

7.  **Evaluation & Feedback Module:**
    *   **Role:** Assesses the quality of generated text and provides constructive feedback to the user.
    *   **Responsibilities:** Grammatical error detection, stylistic suggestions, coherence checks, and potentially originality scores. It might also allow users to rate outputs to improve future generations.

#### Interactions and Data Flow:

*   The **UI** sends user prompts and existing text to the **NLU Module**.
*   The **NLU Module** processes this input and passes the contextual understanding to the **Text Generation Module**, **Text Reconstruction Module**, and **Translation Module**.
*   These generation/transformation modules query the **Knowledge Base** for relevant information (e.g., character names, specific stylistic rules) to guide their output.
*   The outputs from the generation/transformation modules are sent back to the **UI** for display.
*   Optionally, the generated text can be routed through the **Evaluation & Feedback Module** before being displayed to the user, providing immediate suggestions.
*   User feedback via the **UI** can be used to update the **Knowledge Base** or fine-tune models within the generation modules.
*   **API Connections:** External services like the Gemini API would be accessed by the **Translation Module** and potentially the **Text Generation Module** (for more advanced generative tasks).

'''
#### Diagrammatic Representation: